<a href="https://www.nvidia.com/dli"> <img src="images/DLI_Header.png" alt="Header" style="width: 400px;"/> </a>

# 11.0 Deploying Riva Services within a Kubernetes Cluster and Further Riva API Examples 
## (part of Lab 3)

In this notebook, you'll deploy NVIDIA Riva within Kubernetes, and try some API queries for text-to-speech (TTS) and natural language processing (NLP).

**[11.1 Deploy NVIDIA Riva](#11.1-Deploy-NVIDIA-Riva)<br>**
&nbsp;&nbsp;&nbsp;&nbsp;[11.1.1 Exercise: Configure Helm Values and Deploy](#11.1.1-Exercise:-Configure-Helm-Values-and-Deploy)<br>
**[11.2 Riva Services](#11.2-Riva-Services)<br>**
**[11.3 Riva TTS Example](#11.3-Riva-TTS-Example)<br>**
&nbsp;&nbsp;&nbsp;&nbsp;[11.3.1 Exercise: Pod IP with Port 50051](#11.3.1-Exercise:-Pod-IP-with-Port-50051)<br>
&nbsp;&nbsp;&nbsp;&nbsp;[11.3.2 Exercise: LoadBalancer IP with Port 50051](#11.3.2-Exercise:-LoadBalancer-IP-with-Port-50051)<br>
&nbsp;&nbsp;&nbsp;&nbsp;[11.3.3 Exercise: Localhost with Mapped Port](#11.3.3-Exercise:-Localhost-with-Mapped-Port)<br>
&nbsp;&nbsp;&nbsp;&nbsp;[11.3.4 Upgrade the Service with Helm](#11.3.4-Upgrade-the-Service-with-Helm)<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[11.3.4.1 Exercise: Upgrade the Service Type to NodePort](#11.3.4.1-Exercise:-Upgrade-the-Service-Type-to-NodePort)<br>
&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;[11.3.4.2 Verify the Upgrade](#11.3.4.2-Verify-the-Upgrade)<br>
**[11.4 Riva NLP Examples](#11.4-Riva-NLP-Examples)<br>**
&nbsp;&nbsp;&nbsp;&nbsp;[11.4.1 `analyze_intent` API](#11.4.1-analyze_intent-API)<br>
&nbsp;&nbsp;&nbsp;&nbsp;[11.4.2 `punctuate_text` API](#11.4.2-punctuate_text-API)<br>
&nbsp;&nbsp;&nbsp;&nbsp;[11.4.3 Shutdown](#11.4.3-Shutdown)<br>

In the previous parts of the class, you have deployed Riva using very basic shell commands. 
You have also deployed a basic CUDA application to a Kubernetes cluster.
Now it is time to put it all together and deploy Riva into production!

### Notebook Dependencies
1. The steps in this notebook assume that you are starting with a K8s cluster that is GPU enabled with feature discovery.  Let's ensure that by stopping and restarting the a cluster and bringing it to a known state. 
2. As with earlier NVIDIA Riva deployments, you need NGC API credentials.  In this case, you'll also need your email address.

In [1]:
# Delete and restart K8s
!minikube delete
!minikube start --driver=none
# Install the GPU device plugin with Helm
!helm repo add nvdp https://nvidia.github.io/k8s-device-plugin \
   && helm repo update
!helm upgrade -i nvdp nvdp/nvidia-device-plugin \
  --namespace nvidia-device-plugin \
  --create-namespace \
  --version 0.13.0
# Install GPU feature discovery with Helm
!helm repo add nvgfd https://nvidia.github.io/gpu-feature-discovery \
    && helm repo update
!helm upgrade -i nvgfd nvgfd/gpu-feature-discovery \
  --version 0.7.0 \
  --namespace gpu-feature-discovery \
  --create-namespace

🔄  Uninstalling Kubernetes v1.22.2 using kubeadm ...
🔥  Deleting "minikube" in none ...
💀  Removed all traces of the "minikube" cluster.
😄  minikube v1.23.2 on Ubuntu 20.04 (docker/amd64)
✨  Using the none driver based on user configuration
👍  Starting control plane node minikube in cluster minikube
🤹  Running on localhost (CPUs=4, Memory=15818MB, Disk=297738MB) ...
ℹ️  OS release is Ubuntu 20.04.5 LTS
🐳  Preparing Kubernetes v1.22.2 on Docker 23.0.6 ...
    ▪ Generating certificates and keys ...
    ▪ Booting up control plane ...
    ▪ Configuring RBAC rules ...
🤹  Configuring local host environment ...

❗  The 'none' driver is designed for experts who need to integrate with an existing VM
💡  Most users should use the newer 'docker' driver instead, which does not require root!
📘  For more information, see: https://minikube.sigs.k8s.io/docs/reference/drivers/none/

❗  kubectl and minikube configuration will be stored in /root
❗  To use kubectl or minikube commands as your own user, you

In [5]:
# Fill in your personal API key and email address (valid in the scope of this notebook)
NGC_API_KEY = "OG0xZGprcmlia2Exdm92NWh0YWFqMTFjdDQ6NmFkMjI1NzYtMjUxNi00OTMyLTliMjctYTgzZDE1N2U5ZDVj"
NGC_EMAIL = "daboramidu93@gmail.com"

In [6]:
%%bash
# Copy all the ASR and TTS models for convenience (faster deployment)
# Time is about 1-2 minutes for the copy unless already done previously 
cp -rn  /dli_workspace/riva-asr-model-repo/* \
    /dli_workspace/riva-full-model-repo/
cp -rn  /dli_workspace/riva-tts-model-repo/* \
    /dli_workspace/riva-full-model-repo/

---
# 11.1 Deploy NVIDIA Riva

The instructions for deploying NVIDIA Riva on Kubernetes are available on the [NGC Riva Speech Skills Helm chart](https://catalog.ngc.nvidia.com/orgs/nvidia/teams/riva/helm-charts/riva-api) page.

Start by fetching `riva-api` with Helm, and examining the assets downloaded. 

In [7]:
# Fetch riva-api with Helm
!helm fetch https://helm.ngc.nvidia.com/nvidia/riva/charts/riva-api-2.8.1.tgz \
    --username='$oauthtoken' --password=$NGC_API_KEY --untar

Error: failed to untar: a file or directory with the name riva-api-2.8.1.tgz already exists


In [8]:
!ls -l riva-api

total 16
-rw-r--r-- 1 root root   97 Jul 17 13:48 Chart.yaml
drwxr-xr-x 2 root root 4096 Jul 17 13:48 templates
-rw-r--r-- 1 root root 6441 Jul 17 13:48 values.yaml


The configuration file, `values.yaml` contains a number of settings for the service including image details, credentials, and service type.  It also contains a list of ASR, NLP, and TTS models that will be downloaded and optimized upon initialization under `ngcModelConfigs:`

In [9]:
!cat riva-api/values.yaml

# Copyright (c) 2019, NVIDIA CORPORATION. All rights reserved.
#
# Redistribution and use in source and binary forms, with or without
# modification, are permitted provided that the following conditions
# are met:
#  * Redistributions of source code must retain the above copyright
#    notice, this list of conditions and the following disclaimer.
#  * Redistributions in binary form must reproduce the above copyright
#    notice, this list of conditions and the following disclaimer in the
#    documentation and/or other materials provided with the distribution.
#  * Neither the name of NVIDIA CORPORATION nor the names of its
#    contributors may be used to endorse or promote products derived
#    from this software without specific prior written permission.
#
# THIS SOFTWARE IS PROVIDED BY THE COPYRIGHT HOLDERS ``AS IS'' AND ANY
# EXPRESS OR IMPLIED WARRANTIES, INCLUDING, BUT NOT LIMITED TO, THE
# IMPLIED WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR
# PURPOSE ARE DISCLAIM

The Helm Chart starts two containers:
* `riva-model-init` - Responsible for fetching all of the model assets configured in `values.yaml` and their optimization for the target platform (appropriate TensorRT optimization will be executed).  After initialization is complete, this container will self-terminate.
* `riva-speech-api` - Hosts Riva services after initialization is complete. 

Before proceeding, we'll need to make some edits to set the configurations in `values.yaml` to match our environment and limit the models deployed.  If we deploy all the possible models, we may run out of memory!

In [10]:
# Here is where Riva models are located in our class environment
RIVA_MODEL_REPO = "/dli_workspace/riva-full-model-repo"
!ls -al $RIVA_MODEL_REPO

total 430964
drwxr-xr-x  5 root   root      4096 Jul 17 07:20 .
drwxr-xr-x  8 root   root      4096 Jul 17 09:11 ..
drwxr-xr-x 13 root   root      4096 Jul 17 12:04 artifacts
drwxr-xr-x 49 root   root      4096 Jul 17 12:05 models
-rwxrwxrwx  1 docker 1000 441282560 Jan 23  2023 ner_restaurant.nemo
drwxr-xr-x  2 root   root      4096 Jul 17 12:05 rmir


## 11.1.1 Exercise: Configure Helm Values and Deploy
Modify the YAML file for our environment and deploy `riva-api` with Helm.  For our environment, the host path location for Riva `models`, `rmir`, and `artifacts` is `/dli_workspace/riva-full-model-repo`.  We also need to comment out all of the models listed to avoid unnecessary deployments as we already have the models we need in the `/dli_workspace/riva-full-model-repo` directory.

Exercise:
* Open the [values.yaml](riva-api/values.yaml) config file
* Comment out all uncommented models under `ngcModelConfigs:`
* Modify the `modelDeployVolume.hostPath.path` to reflect our environment
* Modify `artifactDeployVolume.hostPath.path` to reflect our environment
* Save the file
* Check your work against the [solution](solutions/ex11.1.1.yaml) before moving on
* Deploy it!

In [11]:
# TODO modify values.yaml so that this cell verifies changes are correct
# Check your work - your file should have the same uncommented models (none!) and folder paths as the solution!
print("YOUR SETTINGS\n=============")
!cat riva-api/values.yaml | grep -v "^\s*[#;]" | sed -n '/ngcModelConfigs:/,/modelDeployVolume:/p' | sed ';$d'
!cat riva-api/values.yaml | grep -v "^\s*[#;]" | grep -A 20 modelDeployVolume: | grep 'DeployVolume\|path'
print("\nSOLUTION SETTINGS\n=================")
!cat solutions/ex11.1.1.yaml | grep -v "^\s*[#;]" | sed -n '/ngcModelConfigs:/,/modelDeployVolume:/p' | sed ';$d'
!cat solutions/ex11.1.1.yaml | grep -v "^\s*[#;]" | grep -A 20 modelDeployVolume: | grep 'DeployVolume\|path'

YOUR SETTINGS
  ngcModelConfigs:
    asr:

    nlp:
      - rmir_nlp_intent_slot_bert_base
      - rmir_nlp_question_answering_bert_base
      - rmir_nlp_punctuation_bert_base_en_us
      - rmir_nlp_text_classification_bert_base
      - rmir_nlp_named_entity_recognition_bert_base
    tts:
      - rmir_tts_fastpitch_hifigan_en_us_ipa

  modelDeployVolume:
      path: /dli_workspace/riva-full-model-repo
  artifactDeployVolume:
      path: /dli_workspace/riva-full-model-repo

SOLUTION SETTINGS
  ngcModelConfigs:
    asr:

    nlp:
    tts:

  modelDeployVolume:
      path: /dli_workspace/riva-full-model-repo
  artifactDeployVolume:
      path: /dli_workspace/riva-full-model-repo/


In [12]:
# Quick Fix!
!cp solutions/ex11.1.1.yaml riva-api/values.yaml

In [13]:
%env model_key_string=tlt_encode

!helm install riva-api riva-api \
    --set ngcCredentials.password=`echo -n $NGC_API_KEY | base64 -w0` \
    --set ngcCredentials.email=$NGC_EMAIL \
    --set modelRepoGenerator.modelDeployKey=`echo -n model_key_string | base64 -w0`


env: model_key_string=tlt_encode
NAME: riva-api
LAST DEPLOYED: Thu Jul 17 14:09:07 2025
NAMESPACE: default
STATUS: deployed
REVISION: 1
TEST SUITE: None


---
# 11.2 Riva Services

In [14]:
!kubectl describe pods riva-api

Name:         riva-api-7bc88844cd-j9nm5
Namespace:    default
Priority:     0
Node:         d3cb20a8d421/172.18.0.4
Start Time:   Thu, 17 Jul 2025 14:09:07 +0000
Labels:       app=riva-api
              pod-template-hash=7bc88844cd
              release=riva-api
Annotations:  <none>
Status:       Pending
IP:           172.17.0.7
IPs:
  IP:           172.17.0.7
Controlled By:  ReplicaSet/riva-api-7bc88844cd
Init Containers:
  riva-model-init:
    Container ID:  docker://952e2b0fe24d7a989a0025bea95ed6749009e4ece30212c477f9915560415bbd
    Image:         nvcr.io/nvidia/riva/riva-speech:2.8.1-servicemaker
    Image ID:      docker://sha256:02956288f640216c75ca832ae06b844be654284120295ffdd5b5c6f0186fc7ee
    Port:          <none>
    Host Port:     <none>
    Command:
      download_and_deploy_ngc_models
    State:          Running
      Started:      Thu, 17 Jul 2025 14:09:08 +0000
    Ready:          False
    Restart Count:  0
    Limits:
      nvidia.com/gpu:  1
    Requests:
      nvid

At first, the models are downloading (this is reflected in the status), so we have to wait. Wait a minute and look at the status again.

In [16]:
!kubectl describe pods riva-api | grep -A 2 'Containers:\|State:'

Init Containers:
  riva-model-init:
    Container ID:  docker://952e2b0fe24d7a989a0025bea95ed6749009e4ece30212c477f9915560415bbd
--
    State:          Terminated
      Reason:       Completed
      Exit Code:    0
--
Containers:
  riva-speech-api:
    Container ID:  docker://febdbbea5ab9927aca0efbbb465988023c3cc11c4a535cddc13687430117874f
--
    State:          Running
      Started:      Thu, 17 Jul 2025 14:10:56 +0000
    Ready:          False


In [17]:
!kubectl describe pods riva-api | grep riva-api

Name:         riva-api-7bc88844cd-j9nm5
Labels:       app=riva-api
              release=riva-api
Controlled By:  ReplicaSet/riva-api-7bc88844cd
  Normal   Scheduled  2m50s              default-scheduler  Successfully assigned default/riva-api-7bc88844cd-j9nm5 to d3cb20a8d421


We need to wait until the status of the `riva-model-init` container changes from "Waiting" to "Running". You can keep executing the previous command to check as many times as needed.  Once `riva-model-init` is "Running", we should be able to view the Docker container logs. We need the name of the pod to view the logs, which we'll grab with a Linux `grep` command.

In [18]:
%%bash
# Grab the name
RIVA_API_LONGNAME=$(kubectl describe pods riva-api | grep "Name:         riva-api-" | awk '{print $2}')
echo "The pod name is $RIVA_API_LONGNAME"
# Check the logs
kubectl logs $RIVA_API_LONGNAME --container=riva-model-init

The pod name is riva-api-7bc88844cd-j9nm5
/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
/data/artifacts /opt/riva
/opt/riva
/bin/bash: warning: setlocale: LC_ALL: cannot change locale (en_US.UTF-8)
2025-07-17 14:09:10,507 [INFO] Writing Riva model repository to '/data/models'...
2025-07-17 14:09:10,507 [INFO] The riva model repo target directory is /data/models
2025-07-17 14:09:16,687 [INFO] Using obey-precision pass with fp16 TRT
2025-07-17 14:09:16,687 [WARNING] /data/models/riva-trt-riva_ner-nn-bert-base-uncased already exists, skipping deployment.  To force deployment rerun with -f or remove the /data/models/riva-trt-riva_ner-nn-bert-base-uncased
2025-07-17 14:09:16,687 [WARNING] /data/models/token_classification_tokenizer-en-US already exists, skipping deployment.  To force deployment rerun with -f or remove the /data/models/token_classification_tokenizer-en-US
2025-07-17 14:09:16

The logs should say that the models are already deployed and optimized and that the initialization has finished.  For example, they should consist of lines like:

```
2023-01-18 23:14:15,563 [INFO] Writing Riva model repository to '/data/models'...
2023-01-18 23:14:15,563 [INFO] The riva model repo target directory is /data/models
2023-01-18 23:14:23,844 [INFO] Using obey-precision pass with fp16 TRT
2023-01-18 23:14:23,844 [WARNING] /data/models/riva-trt-riva_ner-nn-bert-base-uncased already exists, skipping deployment. 
```
    
Troubleshooting note:<br>
If there is a mistake in the path configuration, then the initialization container will attempt to download all of the assets. The models take approximately 6GB of space and their target-specific optimization is a non-trivial task.  Therefore, this step can take 45+ minutes. If the logs are saying that Riva is downloading models, you can uninstall this helm deployment by executing `!helm uninstall riva-api`, correct the [values.yaml](riva-api/values.yaml) file, and try deploying again. 

When Riva model initialization is complete, Riva services will initialize. This can also take a while as all models need to be loaded to memory and verified, and there are quite a few models!

In [19]:
!ls -l $RIVA_MODEL_REPO/models
!du -sh $RIVA_MODEL_REPO/models

total 188
drwxr-xr-x 3 root root 4096 Jul 17 12:04 conformer-en-US-asr-offline
drwxr-xr-x 3 root root 4096 Jul 17 12:04 conformer-en-US-asr-offline-ctc-decoder-cpu-streaming-offline
drwxr-xr-x 3 root root 4096 Jul 17 12:04 conformer-en-US-asr-offline-endpointing-streaming-offline
drwxr-xr-x 3 root root 4096 Jul 17 12:04 conformer-en-US-asr-offline-feature-extractor-streaming-offline
drwxr-xr-x 3 root root 4096 Jul 17 12:04 conformer-en-US-asr-streaming
drwxr-xr-x 3 root root 4096 Jul 17 12:04 conformer-en-US-asr-streaming-ctc-decoder-cpu-streaming
drwxr-xr-x 3 root root 4096 Jul 17 12:04 conformer-en-US-asr-streaming-endpointing-streaming
drwxr-xr-x 3 root root 4096 Jul 17 12:04 conformer-en-US-asr-streaming-feature-extractor-streaming
drwxr-xr-x 3 root root 4096 Jul 17 12:04 conformer-es-US-asr-offline
drwxr-xr-x 3 root root 4096 Jul 17 12:04 conformer-es-US-asr-offline-ctc-decoder-cpu-streaming-offline
drwxr-xr-x 3 root root 4096 Jul 17 12:04 conformer-es-US-asr-offline-endpointing-s

Check to see if the service container, `riva-speech-api` is running yet.<br>
Once it is, take a look at the logs for the container.  The logs should list all the models loaded and confirm that "Riva Conversational AI Server listening on 0.0.0.0:50051" in the last line.

In [20]:
# Repeat execution of this cell until riva-speech-api "State" is "Running" and "Ready" is "True"
!kubectl describe pods riva-api | grep -A 2 'Containers:\|State:'

Init Containers:
  riva-model-init:
    Container ID:  docker://952e2b0fe24d7a989a0025bea95ed6749009e4ece30212c477f9915560415bbd
--
    State:          Terminated
      Reason:       Completed
      Exit Code:    0
--
Containers:
  riva-speech-api:
    Container ID:  docker://febdbbea5ab9927aca0efbbb465988023c3cc11c4a535cddc13687430117874f
--
    State:          Running
      Started:      Thu, 17 Jul 2025 14:10:56 +0000
    Ready:          True


<h3 style="color:red;">Important!</h3>

Do not continue until execution of the above cell shows the `riva-speech-api` "State" is "Running" and "Ready" is "True".  It should look something like this:

```text
Init Containers:
  riva-model-init:
    Container ID:  docker://b3abc3ca46b51750b6f4f5c8e0f13b015691c209ae80c628c964c587265188b6
--
    State:          Terminated
      Reason:       Completed
      Exit Code:    0
--
Containers:
  riva-speech-api:
    Container ID:  docker://c8927d907d58fae704c6d74e0803f5d522df7bb4b314a2f7681ef453f7a15a92
--
    State:          Running
      Started:      Mon, 08 Apr 2024 20:34:24 +0000
    Ready:          True
```

In [21]:
%%bash
# Grab the name
RIVA_API_LONGNAME=$(kubectl describe pods riva-api | grep "Name:         riva-api-" | awk '{print $2}')

echo "The pod name is $RIVA_API_LONGNAME"
# Check the logs
kubectl logs $RIVA_API_LONGNAME --container=riva-speech-api | tail

The pod name is riva-api-7bc88844cd-j9nm5
W0717 14:11:42.668890   417 normalize.cc:52] Speech Class far file missing:/data/models/conformer-es-US-asr-streaming/1/speech_class.far
I0717 14:11:42.815331   417 model_registry.cc:120] Successfully registered: riva-punctuation-en-US for NLP
I0717 14:11:42.816177   417 model_registry.cc:120] Successfully registered: riva-punctuation-es-US for NLP
I0717 14:11:42.823800   417 model_registry.cc:120] Successfully registered: riva_intent_weather for NLP
I0717 14:11:42.824524   417 model_registry.cc:120] Successfully registered: riva_ner for NLP
I0717 14:11:42.825222   417 model_registry.cc:120] Successfully registered: riva_qa for NLP
I0717 14:11:42.825826   417 model_registry.cc:120] Successfully registered: riva_text_classification_domain for NLP
I0717 14:11:42.846323   417 model_registry.cc:120] Successfully registered: fastpitch_hifigan_ensemble-English-US for TTS
I0717 14:11:42.903069   417 riva_server.cc:171] Riva Conversational AI Server li

---
# 11.3 Riva TTS Example

If you have observed "Riva Conversational AI Server listening on 0.0.0.0:50051" in the logs, we are ready to run an application. We will query the API with a TTS example. More information on the API can be found [in the documentation](https://docs.nvidia.com/deeplearning/riva/user-guide/docs/tutorials/tts-python-basics-and-customization-with-ssml.html)<br>

First, import the dependencies:

In [22]:
import io
import librosa
from time import time
import numpy as np
import IPython.display as ipd
import riva.client 

Configure the connection to our server. As you might recall, the service is listening on port 50051. Lets try configuring localhost:50051.  Call the app and output an audio file to listen to.

In [23]:
auth = riva.client.Auth(uri='localhost:50051')

Next, we'll create a little function that sets the channel and submits a line of text to the `SynthesizeSpeech` model and returns an audio sample.  Then run the audio!

In [24]:
sample_rate_hz = 44100

def test_tts(auth, input_text):
    riva_tts = riva.client.SpeechSynthesisService(auth)    
    req = { 
            "language_code"  : "en-US",
            "encoding"       : riva.client.AudioEncoding.LINEAR_PCM ,   # Currently only LINEAR_PCM is supported
            "sample_rate_hz" : sample_rate_hz,                          # Generate 44.1KHz audio
            "voice_name"     : "English-US.Female-1"                    # The name of the voice to generate
        }
    req["text"] = input_text
    resp = riva_tts.synthesize(**req)
    audio_samples = np.frombuffer(resp.audio, dtype=np.int16)
    return audio_samples

In [25]:
ipd.Audio(test_tts(auth, "Is it recognize speech or wreck a nice beach?"), rate=sample_rate_hz)

_InactiveRpcError: <_InactiveRpcError of RPC that terminated with:
	status = StatusCode.UNAVAILABLE
	details = "failed to connect to all addresses; last error: UNKNOWN: ipv4:127.0.0.1:50051: Failed to connect to remote host: Connection refused"
	debug_error_string = "UNKNOWN:failed to connect to all addresses; last error: UNKNOWN: ipv4:127.0.0.1:50051: Failed to connect to remote host: Connection refused {created_time:"2025-07-17T14:12:49.6101368+00:00", grpc_status:14}"
>

Well, that didn't work... Why?

## 11.3.1 Exercise: Pod IP with Port 50051

When running Riva from within Kubernetes, our "localhost" IP (127.0.0.1) is not connected to the Riva services.  There are a few different pathways we could use to send our request.  The first is to select the Riva API pod IP address and send our requests there. The IP is listed in the pod description. 

In [26]:
!kubectl get pod -o wide 

NAME                        READY   STATUS    RESTARTS   AGE     IP           NODE           NOMINATED NODE   READINESS GATES
riva-api-7bc88844cd-j9nm5   1/1     Running   0          4m39s   172.17.0.7   d3cb20a8d421   <none>           <none>


In [27]:
!kubectl get service --all-namespaces

NAMESPACE               NAME                                  TYPE           CLUSTER-IP       EXTERNAL-IP   PORT(S)                                                        AGE
default                 kubernetes                            ClusterIP      10.96.0.1        <none>        443/TCP                                                        16m
default                 riva-api                              LoadBalancer   10.111.16.191    <pending>     8000:31025/TCP,8001:31325/TCP,8002:31386/TCP,50051:30422/TCP   4m41s
gpu-feature-discovery   nvgfd-node-feature-discovery-master   ClusterIP      10.108.108.124   <none>        8080/TCP                                                       16m
kube-system             kube-dns                              ClusterIP      10.96.0.10       <none>        53/UDP,53/TCP,9153/TCP                                         16m


Replace the `POD_IP` with the actual IP value in the next cell and try it this way.

In [28]:
#TODO replace the POD_IP
auth = riva.client.Auth(uri='172.17.0.7:50051')
ipd.Audio(test_tts(auth, "Is it recognize speech or wreck a nice beach?"), rate=sample_rate_hz)

Did that work?  There is another way as well.  

## 11.3.2 Exercise: LoadBalancer IP with Port 50051

In [29]:
!kubectl get services

NAME         TYPE           CLUSTER-IP      EXTERNAL-IP   PORT(S)                                                        AGE
kubernetes   ClusterIP      10.96.0.1       <none>        443/TCP                                                        18m
riva-api     LoadBalancer   10.111.16.191   <pending>     8000:31025/TCP,8001:31325/TCP,8002:31386/TCP,50051:30422/TCP   7m14s


Alternatively, we could use the load balancer IP that is set up with a 50051 port mapping for requests.  

Replace the `LOADBALANCER_IP` with the actual value in the next cell and try it this way.

In [30]:
#TODO replace the LOADBALANCER_IP
auth = riva.client.Auth(uri='10.111.16.191:50051')
ipd.Audio(test_tts(auth, "Is it recognize speech or wreck a nice beach?"), rate=sample_rate_hz)

## 11.3.3 Exercise: Localhost with Mapped Port
Connect to the external facing port mapped from the load balancer to localhost. In this case, this port is assigned randomly, so lets check what it is by looking at the port mapped to 50051 in the services list:

In [31]:
!kubectl get services

NAME         TYPE           CLUSTER-IP      EXTERNAL-IP   PORT(S)                                                        AGE
kubernetes   ClusterIP      10.96.0.1       <none>        443/TCP                                                        19m
riva-api     LoadBalancer   10.111.16.191   <pending>     8000:31025/TCP,8001:31325/TCP,8002:31386/TCP,50051:30422/TCP   8m2s


Replace the `MAPPED_PORT` with the actual value in the next cell and try it this way.

In [33]:
#TODO replace the MAPPED_PORT
auth = riva.client.Auth(uri='localhost:30422')
ipd.Audio(test_tts(auth, "Is it recognize speech or wreck a nice beach?"), rate=sample_rate_hz)

## 11.3.4 Upgrade the Service with Helm
Load balancing is used to distribute tasks over a set of compute resources.  Since we have just one GPU and pod in our example, we do not need the load balancer.  We can turn it off by changing the service type in the `values.yaml` file executing the upgrade command. Here's what we have now:

In [34]:
!cat ./riva-api/values.yaml | grep -A 4 service:

service:
  type: LoadBalancer
  # configure `type: ClusterIP` in case of 'nginx'
  # type: ClusterIP
  nodeport: 32222


The [`helm upgrade` command](https://helm.sh/docs/helm/helm_upgrade/) has the form:

```
helm upgrade [RELEASE] [CHART] [flags]
```

   * CHART is the archive location of the `chart.yaml` file, `riva-api`
   * RELEASE is be the specific name of the riva-api service deployed. 
   
RELEASE is listed in the services names, so we can grab it from there. 

In [35]:
%%bash
# Show the RELEASE value
RELEASE=$(kubectl get svc -A | grep "riva-api"| awk '{print $2}')
echo $RELEASE

riva-api


### 11.3.4.1 Exercise: Upgrade the Service Type to NodePort
Modify the YAML file for to change the service type from `LoadBalancer` to `NodePort` and upgrade it with Helm.

Exercise:
* Open the [values.yaml](riva-api/values.yaml) config file
* Modify the "service.type" to "NodePort"
* Save the file
* Check your work against the [solution](solutions/ex11.3.4.1.yaml) before moving on
* Upgrade the service!

In [36]:
# TODO modify values.yaml so that this cell verifies changes are correct
# Check your work - your file should have the values as the solution!
print("YOUR SETTING")
!cat ./riva-api/values.yaml | grep -A 4 service:
print("\nSOLUTION SETTING")
!cat solutions/ex11.3.4.1.yaml | grep -A 4 service:

YOUR SETTING
service:
  type: NodePort
  # configure `type: ClusterIP` in case of 'nginx'
  # type: ClusterIP
  nodeport: 32222

SOLUTION SETTING
service:
  type: NodePort
  # configure `type: ClusterIP` in case of 'nginx'
  # type: ClusterIP
  nodeport: 32222


In [37]:
%%bash
RELEASE=$(kubectl get svc -A | grep "riva-api"| awk '{print $2}')
helm upgrade $RELEASE riva-api

Release "riva-api" has been upgraded. Happy Helming!
NAME: riva-api
LAST DEPLOYED: Thu Jul 17 14:22:24 2025
NAMESPACE: default
STATUS: deployed
REVISION: 2
TEST SUITE: None


### 11.3.4.2 Verify the Upgrade
Since we have configured port 32222 as our NodePort, we should see the change now in our live service list:

In [38]:
!kubectl get services

NAME         TYPE        CLUSTER-IP      EXTERNAL-IP   PORT(S)                                                        AGE
kubernetes   ClusterIP   10.96.0.1       <none>        443/TCP                                                        24m
riva-api     NodePort    10.111.16.191   <none>        8000:31025/TCP,8001:31325/TCP,8002:31386/TCP,50051:32222/TCP   13m


As a consequence, we have a known IP:PORT value to reliably expose the Riva server.

In [39]:
auth = riva.client.Auth(uri='localhost:32222')
ipd.Audio(test_tts(auth, "Is it recognize speech or wreck a nice beach?"), rate=sample_rate_hz)

What did the code actually do? It executed a request to a TTS service transcribing the sentence provided, then generated an audio file with the transcript.

---
# 11.4 Riva NLP Examples

In the TTS example, we used the `SpeechSynthesisService` class to synthesize speech.  We can similarly make a requests with `NLPService` class and we'll try a couple of the NLP examples. Since there are several API tasks available for NLP, lets get a list by inspecting the class.  You can also review the code directly at https://github.com/nvidia-riva/python-clients

In [40]:
import inspect
# List the callable objects.
[method_name for method_name in dir(riva.client.NLPService)
                  if callable(getattr(riva.client.NLPService, method_name)) 
                     and method_name[0] not in ['_']
]

['analyze_entities',
 'analyze_intent',
 'classify_text',
 'classify_tokens',
 'natural_query',
 'punctuate_text',
 'transform_text']

## 11.4.1 `analyze_intent` API
The `analyze_intent` API can be used to query an "intent slot" classifier. If we don't have a specific domain, this API can be leveraged with an additional text classification model to classify the domain of the input query before routing the text to the appropriate intent slot model.

We'll keep things simple and use an example where the domain is known. This example skips execution of the domain classifier
and proceeds directly to the intent slot model for the requested domain.

In [41]:
# create a function to return intent of a string
def test_intent(auth, input_text):
    riva_nlp = riva.client.NLPService(auth)
    response = riva_nlp.analyze_intent(
        input_string = input_text,
        options = riva.client.AnalyzeIntentOptions(lang = 'en-US'))
    return response

auth = riva.client.Auth(uri='localhost:32222')
print(test_intent(auth, "How is the humidity today in San Francisco?"))

intent {
  class_name: "weather.humidity"
  score: 1.0
}
slots {
  token: "today"
  label {
    class_name: "weatherforecastdaily"
    score: 0.9998509883880615
  }
}
slots {
  token: "san francisco ?"
  label {
    class_name: "weatherplace"
    score: 0.9999390244483948
  }
}
domain_str: "weather"
domain {
  class_name: "weather"
  score: 0.9973570108413696
}



In [42]:
# Some weather Intent queries
queries = [
    "Is it currently cloudy in Tokyo?",
    "What is the annual rainfall in Pune?",
    "What is the humidity going to be tomorrow?"
]

auth = riva.client.Auth(uri='localhost:32222')
for q in queries:
    print(q, '\n', test_intent(auth, q))

Is it currently cloudy in Tokyo? 
 intent {
  class_name: "weather.cloudy"
  score: 1.0
}
slots {
  token: "tokyo"
  label {
    class_name: "weatherplace"
    score: 0.9999169707298279
  }
}
slots {
  token: "?"
  label {
    class_name: "weatherplace"
    score: 0.9997299909591675
  }
}
domain_str: "weather"
domain {
  class_name: "weather"
  score: 0.9972990155220032
}

What is the annual rainfall in Pune? 
 intent {
  class_name: "weather.rainfall"
  score: 1.0
}
slots {
  token: "pune"
  label {
    class_name: "weatherplace"
    score: 0.9998739957809448
  }
}
slots {
  token: "?"
  label {
    class_name: "weatherplace"
    score: 0.999770998954773
  }
}
domain_str: "weather"
domain {
  class_name: "weather"
  score: 0.8264579772949219
}

What is the humidity going to be tomorrow? 
 intent {
  class_name: "weather.humidity"
  score: 1.0
}
slots {
  token: "tomorrow"
  label {
    class_name: "weatherforecastdaily"
    score: 0.999576985836029
  }
}
slots {
  token: "?"
  label {

## 11.4.2 `punctuate_text` API
We can use this API to run the punctuation and capitalization model as follows:

In [43]:
# create a function to return punctuation for a list of strings
plain_text_strings = [
    "add punctuation to this sentence",
    "do you have any red nvidia shirts",
    "i need one cpu four gpus and lots of memory ",
    "for my new computer it's going to be very cool"
]

def test_punctuation(auth, input_texts):
    riva_nlp = riva.client.NLPService(auth)
    response = riva_nlp.punctuate_text(
        input_strings = input_texts,
        language_code = 'en-US')
    return response.text

punctuated = test_punctuation(auth, plain_text_strings)
print(punctuated)

['Add punctuation to this sentence.', 'Do you have any red Nvidia shirts?', 'I need one Cpu, four Gpus, and lots of memory.', "For my new computer, it's going to be very cool."]


## 11.4.3 Shutdown
Clean up your environment by shutting down Riva and K8s.

In [44]:
# Shut down K8s
!minikube delete
!docker kill $(docker ps -q)
# Check for clean environment - this should be empty
!docker ps

🔄  Uninstalling Kubernetes v1.22.2 using kubeadm ...
🔥  Deleting "minikube" in none ...
💀  Removed all traces of the "minikube" cluster.
"docker kill" requires at least 1 argument.
See 'docker kill --help'.

Usage:  docker kill [OPTIONS] CONTAINER [CONTAINER...]

Kill one or more running containers
CONTAINER ID   IMAGE     COMMAND   CREATED   STATUS    PORTS     NAMES


---
<h2 style="color:green;">Congratulations!</h2>

In this notebook, you have:
- Deployed Riva on K8s
- Queried the TTS API, `SynthesizeSpeechRequest`
- Learned how to access the Riva server from various IP:Port combinations
- Queried the `AnalyzeIntent` and `TextTransform` NLP APIs

Now that you've finished the hands-on portion of the course, you can work on the assessments to test your understanding and obtain a certificate!  Move on to the assessment questions in the course dashboard or the [coding assessment notebook](assessment.ipynb)

<a href="https://www.nvidia.com/dli"> <img src="images/DLI_Header.png" alt="Header" style="width: 400px;"/> </a>